# Problem 4: Chromosomal Abnormality Detection

Use EasyEnsemble, probability calibration, and conformal prediction to classify female-fetus samples into normal, abnormal, and indeterminate categories.

## Data and reproducibility

The participant-level NIPT dataset is not distributed in this public repository. To reproduce the analysis, place an authorized copy at `data/nipt_data.xlsx` using the English schema documented in `data/README.md`.


## Setup


In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA_PATH = REPO_ROOT / "data" / "nipt_data.xlsx"
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "The authorized NIPT dataset is not included in this public repository. "
        "Place it at data/nipt_data.xlsx after reviewing data-use restrictions."
    )


## Model training and conformal decisions


In [ ]:

# data：data/nipt_data.xlsx / sheet='female_data'

import numpy as np, pandas as pd
from math import ceil
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (confusion_matrix, roc_auc_score, average_precision_score)
from sklearn.inspection import permutation_importance

from imblearn.ensemble import EasyEnsembleClassifier


FILE_PATH = DATA_PATH
SHEET_NAME = "female_data"
RANDOM_STATE = 2025
EPS_POS_INIT = 0.05
EPS_NEG_GRID = [0.15, 0.20, 0.25, 0.30]
QC_MIN_QUANTILE = 0.10


M = {
 "id":"maternal_id","age":"maternal_age","gest":"gestational_age","BMI":"maternal_bmi",
 "reads":"raw_read_count","map":"mapping_ratio","dup":"duplicate_ratio",
 "uniq":"unique_read_count","filt":"filtered_read_ratio","GC":"gc_content",
 "Z13":"z13","Z18":"z18","Z21":"z21","ZX":"zx",
 "y13":"T13","y18":"T18","y21":"T21",
 "GC13_chr": "gc13",
 "GC18_chr": "gc18",
 "GC21_chr": "gc21",
 "Xconc": "x_chromosome_fraction",
}


def zscore(s: pd.Series):
    m, sd = s.mean(), s.std(ddof=0)
    return (s - m) / sd if (sd and not np.isnan(sd)) else pd.Series(0.0, index=s.index)

def conformal_q(a: np.ndarray, eps: float):
    a = np.sort(a[~np.isnan(a)])
    if a.size == 0: return np.inf
    k = ceil((a.size + 1) * (1 - eps))
    k = min(max(k, 1), a.size)
    return a[k-1]

def metrics_from_cm(cm):
    tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    sens = tp/(tp+fn) if (tp+fn)>0 else np.nan
    spec = tn/(tn+fp) if (tn+fp)>0 else np.nan
    prec = tp/(tp+fp) if (tp+fp)>0 else np.nan
    f1   = 2*prec*sens/(prec+sens) if (prec+sens)>0 else np.nan
    return sens, spec, prec, f1


df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)


num_cols = [c for c in [
    M["age"], M["BMI"], M["reads"], M["map"], M["dup"], M["uniq"], M["filt"], M["GC"],
    M["Z13"], M["Z18"], M["Z21"], M["ZX"],
    M["GC13_chr"], M["GC18_chr"], M["GC21_chr"], M["Xconc"]
] if c in df.columns]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")


logL  = np.log1p(df.get(M["reads"]))
mapr  = df.get(M["map"])
dupr  = df.get(M["dup"])
filtr = df.get(M["filt"])
GCtot = df.get(M["GC"])
df["QC"] = zscore(logL) + zscore(mapr) - zscore(dupr) - zscore(filtr) - zscore((GCtot - 0.5).abs())
QC_MIN = df["QC"].quantile(QC_MIN_QUANTILE)


df["y_any"] = ((df.get(M["y13"])==1)|(df.get(M["y18"])==1)|(df.get(M["y21"])==1)).astype(int)


if M["GC13_chr"] in df.columns:
    df["d13_gc"] = (df[M["GC13_chr"]] - df[M["GC"]]).abs()
if M["GC18_chr"] in df.columns:
    df["d18_gc"] = (df[M["GC18_chr"]] - df[M["GC"]]).abs()
if M["GC21_chr"] in df.columns:
    df["d21_gc"] = (df[M["GC21_chr"]] - df[M["GC"]]).abs()

df["Z13_2"] = df[M["Z13"]]**2; df["Z18_2"] = df[M["Z18"]]**2; df["Z21_2"] = df[M["Z21"]]**2
df["Zmax"]  = df[[M["Z13"], M["Z18"], M["Z21"]]].abs().max(axis=1)
df["Zsum"]  = df[M["Z13"]] + df[M["Z18"]] + df[M["Z21"]]
df["ZxBMI"] = df[M["ZX"]] * df[M["BMI"]]
df["ZmaxxQC"] = df["Zmax"] * df["QC"]


BASE_X = [M["Z13"],M["Z18"],M["Z21"],"Z13_2","Z18_2","Z21_2","Zmax","Zsum",
          M["ZX"],"ZxBMI","ZmaxxQC", M["Xconc"] if M["Xconc"] in df.columns else None,
          "d13_gc" if "d13_gc" in df.columns else None,
          "d18_gc" if "d18_gc" in df.columns else None,
          "d21_gc" if "d21_gc" in df.columns else None,
          M["BMI"],M["GC"],M["map"],M["dup"],M["filt"],M["reads"]]
X_cols = [c for c in BASE_X if (c is not None and c in df.columns)]


valid = df["y_any"].isin([0,1]) & (~df[X_cols].isna().any(axis=1)) & (df["QC"] >= QC_MIN)
groups = df.get(M["id"]).astype(str).fillna("UNK")
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
fit_idx, cal_idx = next(gss.split(df.index.values, df["y_any"].values, groups=groups))
fit_mask = valid & df.index.isin(fit_idx)
cal_mask = valid & df.index.isin(cal_idx)


med = df.loc[fit_mask, X_cols].median()
for c in X_cols: df[c] = df[c].fillna(med[c])


def run_eec(eps_pos=EPS_POS_INIT, eps_neg=0.15):
    X_fit = df.loc[fit_mask, X_cols].values
    y_fit = df.loc[fit_mask, "y_any"].values

    base = EasyEnsembleClassifier(
        n_estimators=20,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    try:
        clf = CalibratedClassifierCV(base, method="isotonic", cv=5)
        clf.fit(X_fit, y_fit)
    except Exception as e:
        print("Isotonic calibration failed; using sigmoid calibration：", e)
        clf = CalibratedClassifierCV(base, method="sigmoid", cv=5)
        clf.fit(X_fit, y_fit)


    try:
        p_tr = clf.predict_proba(X_fit)[:,1]
        auc = roc_auc_score(y_fit, p_tr) if len(np.unique(y_fit))>1 else np.nan
        ap  = average_precision_score(y_fit, p_tr) if len(np.unique(y_fit))>1 else np.nan
        print(f"[Training reference] AUC={auc:.3f}, AP={ap:.3f}")
    except Exception:
        pass


    s_all = pd.Series(clf.predict_proba(df.loc[:, X_cols].values)[:,1], index=df.index)


    cal_ok = cal_mask & (~s_all.isna())
    y_cal  = df.loc[cal_ok, "y_any"].astype(int).values
    s_cal  = s_all.loc[cal_ok].astype(float).values

    A_pos, A_neg = 1.0 - s_cal[y_cal==1], s_cal[y_cal==0]
    q_pos, q_neg = conformal_q(A_pos, eps_pos), conformal_q(A_neg, eps_neg)


    def decide(s, qc):
        if (qc < QC_MIN) or pd.isna(s): return "Indeterminate - retest"
        G=set()
        if (1.0 - s) <= q_pos: G.add(1)
        if s <= q_neg:         G.add(0)
        return "abnormal" if G=={1} else ("normal" if G=={0} else "Indeterminate - retest")
    decision = pd.Series([decide(s_all[i], df.loc[i,"QC"]) for i in s_all.index], index=s_all.index)


    sub = df.loc[cal_ok].copy()
    sub["dec"] = decision.loc[cal_ok]
    dec = sub.loc[sub["dec"]!="Indeterminate - retest"]
    if len(dec)>0:
        yhat = dec["dec"].map({"abnormal":1,"normal":0}).to_numpy()
        cm = confusion_matrix(dec["y_any"].to_numpy(), yhat, labels=[0,1])
        rec, spec, prec, f1 = metrics_from_cm(cm)
        U_rate = 1 - len(dec)/len(sub)
    else:
        rec=spec=prec=f1=np.nan; U_rate=1.0

    err_pos = np.mean((sub["dec"]=="abnormal") & (sub["y_any"]==0))
    err_neg = np.mean((sub["dec"]=="normal") & (sub["y_any"]==1))

    summ = dict(
        eps_pos=eps_pos, eps_neg=eps_neg, q_pos=q_pos, q_neg=q_neg,
        cal_recall=rec, cal_spec=spec, cal_prec=prec, cal_F1=f1,
        cal_U=U_rate, cal_err_pos=err_pos, cal_err_neg=err_neg, cal_n=int(cal_ok.sum())
    )
    return summ, s_all, decision, clf


rows=[]
for en in EPS_NEG_GRID:
    summ, s_any, dec_any, clf_tmp = run_eec(EPS_POS_INIT, en)
    rows.append(summ)
sweep_tbl = pd.DataFrame(rows).sort_values(["cal_U","cal_spec"], ascending=[True,False]).reset_index(drop=True)
display(sweep_tbl)

def pick_row(tbl):
    ok = (tbl["cal_err_pos"] <= tbl["eps_neg"]) & (tbl["cal_err_neg"] <= tbl["eps_pos"])
    if ok.any():
        return tbl[ok].sort_values(["cal_U","cal_spec"], ascending=[True,False]).iloc[0]
    return tbl.sort_values(["cal_err_pos","cal_U"], ascending=[True,True]).iloc[0]

best_row = pick_row(sweep_tbl)
EPS_POS, EPS_NEG = float(best_row["eps_pos"]), float(best_row["eps_neg"])
print("\nSelected thresholds：", {"eps_pos":EPS_POS, "eps_neg":EPS_NEG})


final_summ, final_score, final_dec, final_clf = run_eec(EPS_POS, EPS_NEG)
print("\n=== Final calibration-set results ===")
print({k: final_summ[k] for k in ["cal_recall","cal_spec","cal_prec","cal_F1","cal_U","cal_err_pos","cal_err_neg","q_pos","q_neg","cal_n"]})

final_df = pd.DataFrame({
    "maternal_id": df[M["id"]],
    "ANY_probability": final_score,
    "Final decision": final_dec,
    "Observed_ANY": df["y_any"]
})
final_df.to_csv(OUTPUT_DIR / "q4_final_predictions_any.csv", index=False, encoding="utf-8-sig")
print("Exported：q4_final_predictions_any.csv")


X_fit = df.loc[fit_mask, X_cols].values
y_fit = df.loc[fit_mask, "y_any"].values
perm = permutation_importance(final_clf, X_fit, y_fit, n_repeats=30, random_state=2025,
                              n_jobs=-1, scoring="average_precision")
imp = (pd.DataFrame({"Feature": X_cols, "Importance": perm.importances_mean})
         .sort_values("Importance", ascending=False).head(10))
display(imp)
imp.to_csv(OUTPUT_DIR / "q4_feature_importance_top10.csv", index=False, encoding="utf-8-sig")


with pd.ExcelWriter(OUTPUT_DIR / "q4_routeB_summary.xlsx", engine="openpyxl") as w:
    sweep_tbl.to_excel(w, sheet_name="eps_neg_sweep", index=False)
    pd.DataFrame([{
        "eps_pos":EPS_POS, "eps_neg":EPS_NEG,
        "q_pos":final_summ["q_pos"], "q_neg":final_summ["q_neg"],
        "cal_recall":final_summ["cal_recall"], "cal_spec":final_summ["cal_spec"],
        "cal_prec":final_summ["cal_prec"], "cal_F1":final_summ["cal_F1"],
        "cal_U":final_summ["cal_U"], "cal_err_pos":final_summ["cal_err_pos"], "cal_err_neg":final_summ["cal_err_neg"],
        "cal_n":final_summ["cal_n"]
    }]).to_excel(w, sheet_name="final_thresholds_and_metrics", index=False)
    imp.to_excel(w, sheet_name="Top10Importance", index=False)

print("\nExported：q4_routeB_summary.xlsx / q4_feature_importance_top10.csv")


## Diagnostic figures


In [ ]:



import numpy as np, pandas as pd, matplotlib.pyplot as plt, os
from sklearn.metrics import confusion_matrix, precision_recall_curve, average_precision_score
from sklearn.inspection import permutation_importance


required = ["df","M","final_score","final_dec","q_pos","q_neg","EPS_POS","EPS_NEG","X_cols"]
missing = [k for k in required if k not in globals()]
if missing:
    raise AssertionError(f"Missing variables：{', '.join(missing)}。Run the modeling cell first to create these variables。")


y_any = df["y_any"] if "y_any" in df.columns else (
    ((df[M["y13"]]==1)|(df[M["y18"]]==1)|(df[M["y21"]]==1)).astype(int)
)
final_dec = pd.Series(final_dec, index=df.index) if not isinstance(final_dec, pd.Series) else final_dec
final_score = pd.Series(final_score, index=df.index) if not isinstance(final_score, pd.Series) else final_score
cal_ok = (df.index.isin(cal_idx)) & (~final_score.isna()) if "cal_idx" in globals() else (~final_score.isna())


fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))

chroms = ["T13","T18","T21"]
counts = [int((df[M["y13"]]==1).sum()), int((df[M["y18"]]==1).sum()), int((df[M["y21"]]==1).sum())]
axes[0].bar(chroms, counts)
for i,c in enumerate(counts): axes[0].text(i, c+max(counts)*0.02, f"n={c}", ha="center", va="bottom")
axes[0].set_title("Positive counts for chromosomes 13, 18, and 21"); axes[0].set_ylabel("Sample count"); axes[0].grid(True, axis="y")

any_counts = y_any.value_counts().reindex([0,1]).fillna(0).astype(int)
axes[1].bar(["ANY=0","ANY=1"], any_counts.values)
for i,c in enumerate(any_counts.values):
    axes[1].text(i, c+any_counts.max()*0.02, f"n={c} ({c/any_counts.sum():.1%})", ha="center", va="bottom")
axes[1].set_title("ANY class imbalance"); axes[1].grid(True, axis="y")
save_fig("fig1_class_structure.png")


def box_with_jitter(ax, data_neg, data_pos, title, ylabel="Z score"):
    bp = ax.boxplot([data_neg, data_pos], positions=[1,2], widths=0.5, patch_artist=True,
                    showfliers=False, medianprops=dict(color="black"))
    for patch, color in zip(bp['boxes'], ["#8FBBD9","#F4A582"]): patch.set_facecolor(color)
    rng = np.random.default_rng(2025)
    for j, arr in enumerate([data_neg, data_pos]):
        x = np.full_like(arr.astype(float), j+1, dtype=float) + rng.normal(0, 0.04, size=len(arr))
        ax.scatter(x, arr, s=10, alpha=0.5)
    ax.axhline(0, color="gray", lw=1, ls="--", alpha=0.8)
    ax.set_xticks([1,2]); ax.set_xticklabels(["Negative","positive"])
    ax.set_title(title); ax.set_ylabel(ylabel); ax.grid(True, axis="y")

fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharey=True)
z13 = pd.to_numeric(df[M["Z13"]], errors="coerce")
box_with_jitter(axes[0], z13[df[M["y13"]]==0].dropna().values, z13[df[M["y13"]]==1].dropna().values, "Z13 distribution by T13 status")
z18 = pd.to_numeric(df[M["Z18"]], errors="coerce")
box_with_jitter(axes[1], z18[df[M["y18"]]==0].dropna().values, z18[df[M["y18"]]==1].dropna().values, "Z18 distribution by T18 status")
z21 = pd.to_numeric(df[M["Z21"]], errors="coerce")
box_with_jitter(axes[2], z21[df[M["y21"]]==0].dropna().values, z21[df[M["y21"]]==1].dropna().values, "Z21 distribution by T21 status")
save_fig("fig2_z_distributions.png")


fig = plt.figure(figsize=(8.2, 4.6)); ax = plt.gca()
mask = cal_ok; y_c = y_any[mask].values; s_c = final_score[mask].values
bins = np.linspace(0,1,41)
ax.hist(s_c[y_c==0], bins=bins, alpha=0.6, density=True, label="Normal (calibration)")
ax.hist(s_c[y_c==1], bins=bins, alpha=0.6, density=True, label="Abnormal (calibration)")
t_low, t_high = q_neg, 1 - q_pos
ax.axvline(t_low,  color="black", lw=1.2, ls="--", label=f"q_neg = {t_low:.3f}")
ax.axvline(t_high, color="black", lw=1.2, ls="-.", label=f"1 - q_pos = {t_high:.3f}")
ax.fill_betweenx([0, ax.get_ylim()[1]], 0, t_low,  color="C0", alpha=0.08, label="Normal decision region")
ax.fill_betweenx([0, ax.get_ylim()[1]], t_low, t_high, color="gray", alpha=0.10, label="Indeterminate region")
ax.fill_betweenx([0, ax.get_ylim()[1]], t_high, 1, color="C1", alpha=0.08, label="Abnormal decision region")
ax.set_xlabel("ANY probability"); ax.set_ylabel("Density"); ax.set_title(f"ANY score distribution (calibration set)  ε_pos={EPS_POS}, ε_neg={EPS_NEG}")
ax.legend(ncol=2); ax.grid(True, axis="y")
save_fig("fig3_any_score_hist_thresholds.png")


fig = plt.figure(figsize=(6.4, 5.0)); ax = plt.gca()
prec, reca, _ = precision_recall_curve(y_c, s_c)
ap = average_precision_score(y_c, s_c) if len(np.unique(y_c))>1 else np.nan
base = (y_c==1).mean() if len(y_c)>0 else 0
ax.plot(reca, prec, lw=2, label=f"PR (AP={ap:.3f})")
ax.hlines(base, 0, 1, colors="gray", linestyles="--", label=f"Random baseline={base:.3f}", alpha=0.8)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title("Calibration-set precision-recall curve")
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.legend(); ax.grid(True)
save_fig("fig4_pr_curve_calibration.png")


fig = plt.figure(figsize=(5.2, 4.8)); ax = plt.gca()
sub = pd.DataFrame({"y": y_any[mask], "dec": final_dec[mask]})
dec = sub[sub["dec"]!="Indeterminate - retest"]
if len(dec)>0:
    yhat = dec["dec"].map({"abnormal":1,"normal":0}).values
    cm = confusion_matrix(dec["y"].values, yhat, labels=[0,1]).astype(float)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm_norm[i,j]:.2f}\n(n={int(cm[i,j])})", ha="center", va="center")
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Predicted normal","Predicted abnormal"]); ax.set_yticklabels(["Observed normal","Observed abnormal"])
    ax.set_title(f"Confusion matrix for determinate calibration cases\nDeterminate proportion={len(dec)/len(sub):.3f}")
    plt.colorbar(im, fraction=0.046, pad=0.4/10)
else:
    ax.text(0.5,0.5,"All calibration cases are indeterminate", ha="center", va="center")
save_fig("fig5_confusion_matrix_adjudicated.png")


assert "final_clf" in globals() and "fit_mask" in globals(), "Missing final_clf or fit_mask。"
X_fit = df.loc[fit_mask, X_cols].values
y_fit = y_any[fit_mask].values
perm = permutation_importance(final_clf, X_fit, y_fit, n_repeats=30, random_state=2025,
                              n_jobs=-1, scoring="average_precision")
imp_df = (pd.DataFrame({"Feature": X_cols, "Importance": perm.importances_mean})
            .sort_values("Importance", ascending=False)
            .head(10)).iloc[::-1]

fig = plt.figure(figsize=(7.2, 4.8)); ax = plt.gca()
ax.barh(imp_df["Feature"], imp_df["Importance"])
for i,(v,_) in enumerate(zip(imp_df["Importance"], imp_df["Feature"])): ax.text(v, i, f" {v:.3f}", va="center")
ax.set_xlabel("Permutation importance (decrease in AP)"); ax.set_title("Feature Top-10（Training set）"); ax.grid(True, axis="x")
save_fig("fig6_permutation_importance_top10.png")

print("✅ Generated and saved six main figures to：figs_q4/")
